# 09 · Connect a public feed without hiding failures

**35 minutes · Intermediate.** Fetch an optional public USGS feed, retain provenance, handle failures visibly and distinguish live content from saved training examples.

[Run this lab in JupyterLite](https://jltobias.github.io/JupyterLite-World-Monitor-Prototype/lab/index.html?path=09_Live_Data.ipynb) · [Book home](https://jltobias.github.io/JupyterLite-World-Monitor-Prototype/book/intro.html)

**Start:** run cells from top to bottom (Shift+Enter), or use Run → Run All Cells. The first launch downloads the Python runtime and packages. Core analysis uses bundled training data; internet is needed for initial setup and interactive JavaScript. Each lab starts independently.

In [ ]:
import sys
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "pandas", "matplotlib", "folium==0.20.0", "plotly==6.3.0"])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from fieldguide import setup_style, load_tables, summarize, TRAINING
setup_style()
print(TRAINING)

## Browser constraints are part of the architecture
Browser Python uses HTTP endpoints permitted by CORS; it cannot bypass provider policy. Secret API keys do not belong in a public notebook. Use a controlled backend for authenticated production services. This optional example uses the public USGS M4.5+ past-week GeoJSON feed and requires no credential.
Default mode is synthetic. Set `USE_LIVE=True` explicitly to make a request. A failed request warns and returns data labeled **SYNTHETIC TRAINING — LIVE FETCH FAILED**. It never silently presents a fixture as live.

In [ ]:
from worldmonitor import earthquake_feed
from fieldguide import earthquake_table
USE_LIVE=False
feed,provenance=await earthquake_feed(live=USE_LIVE)
display(pd.Series(provenance,name="Provenance"))
events=earthquake_table(feed)
print("Valid rows:",len(events),"Skipped malformed rows:",events.attrs["skipped_records"])
display(events)

## Retain event time and source-generation time
USGS provides time in Unix milliseconds and coordinates in longitude, latitude, depth order. Depth is in kilometers. Magnitude is not impact: exposure, vulnerability, distance and local conditions matter. An empty valid feed should produce zero rows, not a fabricated event.

In [ ]:
generated=provenance.get("generated_ms")
if generated is not None:
    print("Feed generated (UTC):",pd.to_datetime(generated,unit="ms",utc=True))
event_times=[f.get("properties",{}).get("time") for f in feed["features"]]
times=pd.to_datetime(pd.Series(event_times,dtype="float64"),unit="ms",utc=True,errors="coerce")
print("Earliest event time:",times.min(),"Latest:",times.max())
fig,ax=plt.subplots(figsize=(10,4))
if len(events):
    ax.hist(events.magnitude,bins=8,color="#007f86",edgecolor="white")
else:
    ax.text(.5,.5,"No valid events in this response",ha="center",transform=ax.transAxes)
ax.set(xlabel="Magnitude",ylabel="Event count",title=provenance["mode"]+" · Earthquake feed")
plt.show()

In [ ]:
from fieldguide import download_link
import json
export={"provenance":provenance,"geojson":feed}
display(download_link(json.dumps(export,indent=2),"hazard-feed-with-provenance.json","application/json"))

## Production extension pattern
`Browser notebook → authorized backend → provider API`.
The backend holds secrets and implements quotas, caching, access controls and allowed origins. Publish only an appropriately aggregated, licensed response. JupyterLite is not a shared database or an incident-management authority. Record schema changes and freshness checks, and retain a reproducible snapshot alongside a briefing.

## Practice and debrief

Run once in synthetic mode, then optionally in live mode. Compare provenance, not just event counts. Explain the difference between event time, feed-generation time and retrieval time. Never join this global hazard feed to fictional clinic data as if it represented observed impacts.

## Sources and further learning

[USGS GeoJSON feed specification](https://earthquake.usgs.gov/earthquakes/feed/v1.0/geojson.php) · [Pyodide HTTP API](https://pyodide.org/en/stable/usage/api/python-api/http.html) · [World Monitor sandbox](https://www.worldmonitor.app/docs/sandbox).